In [ ]:
!pip install gdown -q

In [ ]:
!pip install tensorflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 742.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00


In [1]:
!gdown --id 1SSrcU-0Ph-ONr5r0qpr9V5ssF-OIVN2r

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1SSrcU-0Ph-ONr5r0qpr9V5ssF-OIVN2r
From (redirected): https://drive.google.com/uc?id=1SSrcU-0Ph-ONr5r0qpr9V5ssF-OIVN2r&confirm=t&uuid=ca956fb4-9ca4-479c-9e45-b90f02e7f544
To: /content/Dataset_Evolutivo_Simulado.zip
100% 627M/627M [00:15<00:00, 40.2MB/s]


In [2]:
!unzip -q Dataset_Evolutivo_Simulado.zip

In [3]:
!pip list

Package                                  Version
---------------------------------------- --------------------
absl-py                                  1.4.0
accelerate                               1.11.0
access                                   1.1.9
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.2
aiosignal                                1.4.0
aiosqlite                                0.21.0
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.17.2
altair                                   5.5.0
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                                    4.11.0
anywidget                          

Si el resultado anterior indica que se detectó una GPU, TensorFlow la utilizará automáticamente para el entrenamiento. Para monitorear la utilización de la GPU durante el entrenamiento, puedes abrir una terminal en Colab (Ctrl+`) y ejecutar `nvidia-smi`.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import time
from tqdm import tqdm
import imageio

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.19.0


In [ ]:
print("Paso 0: Verificando la disponibilidad de la GPU...")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Se encontraron las siguientes GPUs: {gpus}")
    print("TensorFlow ha detectado y está utilizando la GPU.")
    # Puedes configurar el crecimiento de la memoria si tienes problemas de OOM
    # for gpu in gpus:
    #     tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No se encontraron GPUs. TensorFlow se ejecutará en CPU.")

print(f"¿Está TensorFlow usando una GPU? {tf.test.is_gpu_available()}")

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


Paso 0: Verificando la disponibilidad de la GPU...
Se encontraron las siguientes GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow ha detectado y está utilizando la GPU.
¿Está TensorFlow usando una GPU? True


In [ ]:
print("Paso 1: Definiendo parámetros...")
#Donde train_A es la etiqueta de estado temporal inicial y train_B es el estado temporal futuro que se espera como prediccion visual

PATH = 'Preprocesado_Pix2Pix/'
TRAIN_A_DIR = os.path.join(PATH, 'train_A')
TRAIN_B_DIR = os.path.join(PATH, 'train_B')

OUTPUT_IMG_DIR = 'Resultados_Entrenamiento'
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)

BUFFER_SIZE = 400
BATCH_SIZE = 1
IMG_WIDTH = 256
IMG_HEIGHT = 256
CHECKPOINT_INTERVAL = 15

EPOCHS = 100
LAMBDA = 100

Paso 1: Definiendo parámetros...


In [ ]:
import os

print("Updating checkpoint directory to Google Drive...")
drive_checkpoint_dir = '/content/drive/MyDrive/Upao/9Ciclo/DeepLearning/Proyecto/training/Checkpoint-GAN-Simple'
os.makedirs(drive_checkpoint_dir, exist_ok=True)
checkpoint_dir = drive_checkpoint_dir
print(f"Checkpoint directory updated to: {checkpoint_dir}")

print(f"Paso 0: Iniciando el bucle de entrenamiento por {EPOCHS} epochs...")
print(f"Los resultados se guardarán en: {OUTPUT_IMG_DIR}")
print(f"Los checkpoints se guardarán en: {checkpoint_dir}")

Updating checkpoint directory to Google Drive...
Checkpoint directory updated to: /content/drive/MyDrive/Upao/9Ciclo/DeepLearning/Proyecto/training/Checkpoint-GAN-Simple
Paso 0: Iniciando el bucle de entrenamiento por 100 epochs...
Los resultados se guardarán en: Resultados_Entrenamiento
Los checkpoints se guardarán en: /content/drive/MyDrive/Upao/9Ciclo/DeepLearning/Proyecto/training/Checkpoint-GAN-Simple


In [ ]:
print("Paso 2: Definiendo funciones de carga de datos...")

def load_image(image_file):
    """Carga una imagen y la divide en T0 (entrada) y T1 (real)."""
    real_image_file = tf.strings.regex_replace(image_file, "train_A", "train_B")
    input_image = tf.io.read_file(image_file)
    input_image = tf.io.decode_jpeg(input_image, channels=3)
    real_image = tf.io.read_file(real_image_file)
    real_image = tf.io.decode_jpeg(real_image, channels=3)
    input_image = tf.cast(input_image, tf.float32)
    real_image = tf.cast(real_image, tf.float32)
    return input_image, real_image

Paso 2: Definiendo funciones de carga de datos...


In [ ]:
def resize(input_image, real_image, height, width):
    """Redimensiona ambas imágenes."""
    input_image = tf.image.resize(input_image, [height, width],
                                  method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    real_image = tf.image.resize(real_image, [height, width],
                                 method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    return input_image, real_image

In [ ]:
def random_crop(input_image, real_image):
    """Recorte aleatorio a 256x256 (aumentación de datos)."""
    stacked_image = tf.stack([input_image, real_image], axis=0)
    cropped_image = tf.image.random_crop(
        stacked_image, size=[2, IMG_HEIGHT, IMG_WIDTH, 3])
    return cropped_image[0], cropped_image[1]

def normalize(input_image, real_image):
    """Normaliza las imágenes al rango [-1, 1]."""
    input_image = (input_image / 127.5) - 1
    real_image = (real_image / 127.5) - 1
    return input_image, real_image

@tf.function()
def random_jitter(input_image, real_image):
    """Aumentación de datos estándar de Pix2Pix."""

    input_image, real_image = resize(input_image, real_image, 286, 286)

    input_image, real_image = random_crop(input_image, real_image)

    if tf.random.uniform(()) > 0.5:
        input_image = tf.image.flip_left_right(input_image)
        real_image = tf.image.flip_left_right(real_image)

    return input_image, real_image

def load_image_train(image_file):
    """Carga, aplica jitter y normaliza la imagen de entrenamiento."""
    input_image, real_image = load_image(image_file)
    input_image, real_image = random_jitter(input_image, real_image)
    input_image, real_image = normalize(input_image, real_image)
    return input_image, real_image
def load_image_test(image_file):
    """Carga y normaliza una imagen de prueba (sin jitter)."""
    input_image, real_image = load_image(image_file)
    input_image, real_image = resize(input_image, real_image, IMG_HEIGHT, IMG_WIDTH)
    input_image, real_image = normalize(input_image, real_image)
    return input_image, real_image

In [ ]:
print("Paso 3: Creando pipeline de tf.data...")

train_files_pattern = os.path.join(TRAIN_A_DIR, '*.jpg')
num_train_files = len(glob.glob(train_files_pattern))

if num_train_files == 0:
    print(f"¡ADVERTENCIA! No se encontraron imágenes en: {train_files_pattern}")
    print("Asegúrate de haber ejecutado el script de preprocesamiento anterior.")
else:
    print(f"Total de pares de entrenamiento encontrados: {num_train_files}")

train_dataset = tf.data.Dataset.list_files(train_files_pattern)
train_dataset = train_dataset.map(load_image_train,
                                  num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(BUFFER_SIZE)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.list_files(train_files_pattern)
test_dataset = test_dataset.take(10)
test_dataset = test_dataset.map(load_image_test)
test_dataset = test_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

Paso 3: Creando pipeline de tf.data...
Total de pares de entrenamiento encontrados: 1956


In [ ]:
print("Paso 4: Construyendo el Generador (U-Net)...")

def downsample(filters, size, apply_batchnorm=True):
    """Bloque de codificación (Downsampling) en U-Net."""
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(
        layers.Conv2D(filters, size, strides=2, padding='same',
                             kernel_initializer=initializer, use_bias=False))
    if apply_batchnorm:
        result.add(layers.BatchNormalization())
    result.add(layers.LeakyReLU())
    return result

def upsample(filters, size, apply_dropout=False):
    """Bloque de decodificación (Upsampling) en U-Net."""
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(
        layers.Conv2DTranspose(filters, size, strides=2,
                                      padding='same',
                                      kernel_initializer=initializer,
                                      use_bias=False))
    result.add(layers.BatchNormalization())
    if apply_dropout:
        result.add(layers.Dropout(0.5))
    result.add(layers.ReLU())
    return result

def build_generator():
    """Construye el Generador U-Net completo."""
    inputs = layers.Input(shape=[IMG_WIDTH, IMG_HEIGHT, 3])

    # Encoder
    down_stack = [
        downsample(64, 4, apply_batchnorm=False),
        downsample(128, 4),
        downsample(256, 4),
        downsample(512, 4),
        downsample(512, 4),
        downsample(512, 4),
        downsample(512, 4),
        downsample(512, 4),
    ]

    # Decoder
    up_stack = [
        upsample(512, 4, apply_dropout=True),
        upsample(512, 4, apply_dropout=True),
        upsample(512, 4, apply_dropout=True),
        upsample(512, 4),
        upsample(256, 4),
        upsample(128, 4),
        upsample(64, 4),
    ]

    initializer = tf.random_normal_initializer(0., 0.02)
    last = layers.Conv2DTranspose(3, 4,
                                  strides=2,
                                  padding='same',
                                  kernel_initializer=initializer,
                                  activation='tanh')

    x = inputs


    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = reversed(skips[:-1])


    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])

    x = last(x)

    return Model(inputs=inputs, outputs=x)

generator = build_generator()
#tf.keras.utils.plot_model(generator, show_shapes=True, dpi=64)

Paso 4: Construyendo el Generador (U-Net)...


In [ ]:
print("Paso 5: Construyendo el Discriminador (PatchGAN)...")

def build_discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)


    inp = layers.Input(shape=[IMG_WIDTH, IMG_HEIGHT, 3], name='input_image')

    tar = layers.Input(shape=[IMG_WIDTH, IMG_HEIGHT, 3], name='target_image')


    x = layers.concatenate([inp, tar])

    down1 = downsample(64, 4, False)(x)
    down2 = downsample(128, 4)(down1)
    down3 = downsample(256, 4)(down2)


    zero_pad1 = layers.ZeroPadding2D()(down3)
    conv = layers.Conv2D(512, 4, strides=1,
                          kernel_initializer=initializer,
                          use_bias=False)(zero_pad1)

    batchnorm1 = layers.BatchNormalization()(conv)
    leaky_relu = layers.LeakyReLU()(batchnorm1)


    zero_pad2 = layers.ZeroPadding2D()(leaky_relu)

    last = layers.Conv2D(1, 4, strides=1,
                          kernel_initializer=initializer)(zero_pad2)


    return Model(inputs=[inp, tar], outputs=last)

discriminator = build_discriminator()
#tf.keras.utils.plot_model(discriminator, show_shapes=True, dpi=64)

Paso 5: Construyendo el Discriminador (PatchGAN)...


In [ ]:
print("Paso 6: Definiendo pérdidas y optimizadores...")

loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(disc_real_output, disc_generated_output):
    """Pérdida del Discriminador."""

    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)


    generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)

    total_disc_loss = real_loss + generated_loss
    return total_disc_loss

def generator_loss(disc_generated_output, gen_output, target):
    """Pérdida del Generador."""

    gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)

    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))

    total_gen_loss = gan_loss + (LAMBDA * l1_loss)

    return total_gen_loss, gan_loss, l1_loss

generator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)


checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

Paso 6: Definiendo pérdidas y optimizadores...


In [ ]:
print("Paso 6.5: Intentando cargar el último checkpoint...")

# Intenta restaurar el último checkpoint si existe
latest_ckpt = tf.train.latest_checkpoint(checkpoint_dir)
start_epoch = 0

if latest_ckpt:
    checkpoint.restore(latest_ckpt)
    # Extrae el número de época del nombre del checkpoint (ej: "ckpt-15")
    start_epoch = int(latest_ckpt.split('-')[-1])
    print(f"Checkpoint restaurado. Continuando desde la época {start_epoch}")
else:
    print("Iniciando entrenamiento desde cero.")

Paso 6.5: Intentando cargar el último checkpoint...
Iniciando entrenamiento desde cero.


In [ ]:
print("Paso 7: Definiendo función de visualización...")

def generate_images(model, test_input, tar, epoch):
    """Genera y guarda una imagen de muestra."""
    prediction = model(test_input, training=True)

    plt.figure(figsize=(15, 5))

    display_list = [test_input[0], tar[0], prediction[0]]
    title = ['Input Image (T0)', 'Ground Truth (T1)', 'Predicted Image (T_pred)']

    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.title(title[i])
        plt.imshow(display_list[i] * 0.5 + 0.5)
        plt.axis('off')

    save_path = os.path.join(OUTPUT_IMG_DIR, f'image_at_epoch_{epoch:04d}.png')
    plt.savefig(save_path)
    plt.close()

Paso 7: Definiendo función de visualización...


In [ ]:
print("Paso 8: Definiendo el paso de entrenamiento (Pix2Pix)...")
@tf.function
def train_step(input_image, target, epoch):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)

        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        gen_total_loss, gen_gan_loss, gen_l1_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    generator_gradients = gen_tape.gradient(gen_total_loss, generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

    return disc_loss, gen_total_loss, gen_gan_loss, gen_l1_loss

Paso 8: Definiendo el paso de entrenamiento (Pix2Pix)...


In [ ]:
print("Paso 9: Iniciando el bucle de entrenamiento (Corregido)...")

def train(dataset, epochs, start_epoch):

    if num_train_files == 0:
        print("¡ERROR! No se encontraron imágenes en el directorio de entrenamiento.")
        return

    try:
        test_batch = next(iter(test_dataset))
    except StopIteration:
        print("¡ERROR! El dataset de prueba está vacío. No se puede generar imagen de muestra.")
        return

    seed_input, seed_target = test_batch
    print("Usando un lote de prueba fijo para generar imágenes de muestra.")

    for epoch in range(start_epoch, epochs):

        epoch_start_time = time.time()

        print(f"\nGenerando imagen de muestra para la época {epoch + 1}...")
        generate_images(generator, seed_input, seed_target, epoch)

        print(f"Iniciando Época {epoch + 1}/{epochs}")

        pbar = tqdm(total=num_train_files // BATCH_SIZE, desc=f"Época {epoch + 1}", unit="batch")

        epoch_disc_loss = []
        epoch_gen_total_loss = []
        epoch_gen_gan_loss = []
        epoch_gen_l1_loss = []

        for n, (input_image, target) in dataset.enumerate():


            disc_loss, gen_total_loss, gen_gan_loss, gen_l1_loss = train_step(input_image, target, epoch)


            epoch_disc_loss.append(disc_loss.numpy())
            epoch_gen_total_loss.append(gen_total_loss.numpy())
            epoch_gen_gan_loss.append(gen_gan_loss.numpy())
            epoch_gen_l1_loss.append(gen_l1_loss.numpy())

            pbar.update(1)
            if (n + 1) % 50 == 0:
                pbar.set_postfix_str(f"D_loss: {np.mean(epoch_disc_loss):.4f}, G_loss_total: {np.mean(epoch_gen_total_loss):.4f}")

        pbar.close()

        print(f"\n--- Resumen Época {epoch + 1} ---")
        print(f"Tiempo: {time.time() - epoch_start_time:.2f} seg")
        print(f"Pérdida Discriminador (Media): {np.mean(epoch_disc_loss):.4f}")
        print(f"Pérdida Generador (Total):     {np.mean(epoch_gen_total_loss):.4f}")
        print(f"Pérdida Generador (GAN):       {np.mean(epoch_gen_gan_loss):.4f}")
        print(f"Pérdida Generador (L1):        {np.mean(epoch_gen_l1_loss):.4f}")
        print("---------------------------\n")

        # Guardar Checkpoint
        if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
            checkpoint.save(file_prefix=os.path.join(checkpoint_dir, "ckpt"))
            print(f'Checkpoint guardado para la época {epoch + 1}')

    print(f"Generando imagen final del entrenamiento (Época {epochs})...")
    generate_images(generator, seed_input, seed_target, epochs)


Paso 9: Iniciando el bucle de entrenamiento (Corregido)...


In [ ]:
print("Iniciando entrenamiento...")
train(train_dataset, EPOCHS, start_epoch)
print("Entrenamiento completado.")

Iniciando entrenamiento...
Usando un lote de prueba fijo para generar imágenes de muestra.

Generando imagen de muestra para la época 1...
Iniciando Época 1/100


Época 1: 100%|██████████| 1956/1956 [01:44<00:00, 18.70batch/s, D_loss: 0.8864, G_loss_total: 21.3136]



--- Resumen Época 1 ---
Tiempo: 108.65 seg
Pérdida Discriminador (Media): 0.8854
Pérdida Generador (Total):     21.3167
Pérdida Generador (GAN):       1.5314
Pérdida Generador (L1):        0.1979
---------------------------


Generando imagen de muestra para la época 2...
Iniciando Época 2/100


Época 2: 100%|██████████| 1956/1956 [01:38<00:00, 19.86batch/s, D_loss: 0.8171, G_loss_total: 20.8110]



--- Resumen Época 2 ---
Tiempo: 98.97 seg
Pérdida Discriminador (Media): 0.8170
Pérdida Generador (Total):     20.8153
Pérdida Generador (GAN):       1.6824
Pérdida Generador (L1):        0.1913
---------------------------


Generando imagen de muestra para la época 3...
Iniciando Época 3/100


...


--- Resumen Época 98 ---
Tiempo: 98.93 seg
Pérdida Discriminador (Media): 0.8477
Pérdida Generador (Total):     13.2925
Pérdida Generador (GAN):       1.7320
Pérdida Generador (L1):        0.1156
---------------------------


Generando imagen de muestra para la época 99...
Iniciando Época 99/100


Época 99: 100%|██████████| 1956/1956 [01:38<00:00, 19.86batch/s, D_loss: 0.8393, G_loss_total: 13.2661]



--- Resumen Época 99 ---
Tiempo: 98.95 seg
Pérdida Discriminador (Media): 0.8381
Pérdida Generador (Total):     13.2712
Pérdida Generador (GAN):       1.7625
Pérdida Generador (L1):        0.1151
---------------------------


Generando imagen de muestra para la época 100...
Iniciando Época 100/100


Época 100: 100%|██████████| 1956/1956 [01:38<00:00, 19.88batch/s, D_loss: 0.8556, G_loss_total: 13.1495]



--- Resumen Época 100 ---
Tiempo: 98.88 seg
Pérdida Discriminador (Media): 0.8549
Pérdida Generador (Total):     13.1492
Pérdida Generador (GAN):       1.7499
Pérdida Generador (L1):        0.1140
---------------------------

Generando imagen final del entrenamiento (Época 100)...
Entrenamiento completado.


In [ ]:
print("Guardando el modelo generador en formato .keras...")

# Define la ruta para guardar el modelo
model_save_path = os.path.join(checkpoint_dir, 'generator_final.keras')

# Guarda el modelo en formato .keras
generator.save(model_save_path)

print(f"Modelo generador guardado exitosamente en: {model_save_path}")

Guardando el modelo generador en formato .keras...
Modelo generador guardado exitosamente en: /content/drive/MyDrive/Upao/9Ciclo/DeepLearning/Proyecto/training/Checkpoint-GAN-Simple/generator_final.keras


In [ ]:
import tensorflow as tf
import os

print("Cargando el modelo generador desde formato .keras...")

# Define la ruta donde se guardó el modelo (asegúrate de que coincida con la ruta de guardado)
# Si la variable 'checkpoint_dir' no está definida en este contexto, cámbiala por la ruta literal.
model_load_path = os.path.join(checkpoint_dir, 'generator_final.keras')

# Carga el modelo
try:
    loaded_generator = tf.keras.models.load_model(model_load_path)
    print(f"Modelo generador cargado exitosamente desde: {model_load_path}")
    # Puedes probar el modelo cargado si es necesario, por ejemplo:
    # print(loaded_generator.summary())
except Exception as e:
    print(f"Error al cargar el modelo: {e}")

Cargando el modelo generador desde formato .keras...
Modelo generador cargado exitosamente desde: /content/drive/MyDrive/Upao/9Ciclo/DeepLearning/Proyecto/training/Checkpoint-GAN-Simple/generator_final.keras
